# Map of Italian Science — Organisation-Level Citation Analysis

## Research questions addressed in this notebook

- **Which external organisations cite the publications of each Italian institution** recorded in OpenCitations (incoming citations), and at what volume?
- **Which external organisations are cited by each Italian institution** (outgoing citations), and at what volume?
- **How symmetric or asymmetric** is the citation exchange between each Italian institution and its major partners — and does this vary systematically across institutions?

The six Italian institutions examined are: University of Bologna (UNIBO), University of Milan (UNIMI), University of Padua (UNIPD), University of Turin (UNITO), University of Eastern Piedmont (UPO), and Scuola Normale Superiore (SNS).

*This notebook is the organisation-level companion to the country-level analysis. It follows the same data pipeline and exclusion logic, operating at partner-institution granularity rather than country granularity.*

---

## Document structure

| Section | Content |
|---|---|
| **1. Setup & Data Loading** | Imports, configuration constants, country-name standardisation, data loading pipeline |
| **2. Per-Institution Analysis** | Combined butterfly chart (top-15 partners, all six institutions) · Interactive reciprocity scatter |
| **3. Cross-Institution Comparison** | Proportional partner mix across all six institutions |
| **4. Summary of Findings** | Synthesis of key patterns |


## Configuration

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.0f}'.format)

BASE_PATH = Path("../data/citation_counts")

INSTITUTIONS = {
    "UNIBO": BASE_PATH / "UNIBO",
    "UNIMI": BASE_PATH / "UNIMI",
    "UNIPD": BASE_PATH / "UNIPD",
    "UNITO": BASE_PATH / "UNITO",
    "UPO":   BASE_PATH / "UPO",
    "SNS":   BASE_PATH / "SNS",
}
INSTITUTION_LABELS = {
    "UNIBO": "University of Bologna",
    "UNIMI": "University of Milan",
    "UNIPD": "University of Padua",
    "UNITO": "University of Turin",
    "UPO":   "University of Eastern Piedmont",
    "SNS":   "Scuola Normale Superiore",
}
# Legal name as it appears in the CSV — used to exclude only the focal institution
INSTITUTION_SELF_NAMES = {
    "UNIBO": "University of Bologna",
    "UNIMI": "University of Milan",
    "UNIPD": "University of Padua",
    "UNITO": "University of Turin",
    "UPO":   "University of Eastern Piedmont",
    "SNS":   "Scuola Normale Superiore",
}
COLORS = {"incoming": "#B7990D", "outgoing": "#320E3B"}
TOP_N     = 15
RECIP_TOP = 500

COUNTRY_COLORS = {
    "United States":    "#1f77b4",
    "France":           "#B7990D",
    "United Kingdom":   "#d62728",
    "Germany":          "#2ca02c",
    "China":            "#ff7f0e",
    "Spain":            "#9467bd",
    "Japan":            "#e377c2",
    "Canada":           "#17becf",
    "Australia":        "#bcbd22",
    "The Netherlands":  "#8c564b",
    "Switzerland":      "#aec7e8",
    "Russia":           "#c5b0d5",
    "India":            "#ffbb78",
    "South Korea":      "#98df8a",
    "Brazil":           "#ff9896",
    "Poland":           "#f7b6d2",
    "Belgium":          "#c49c94",
    "Türkiye":          "#dbdb8d",
    "Sweden":           "#9edae5",
    "Finland":          "#393b79",
    "Denmark":          "#637939",
    "Taiwan":           "#8c6d31",
    "Greece":           "#843c39",
    "Portugal":         "#7b4173",
    "Austria":          "#5254a3",
    "Italy":            "#e6550d",
}
OTHER_COUNTRY_COLOR = "#cccccc"

# Countries to show in the reciprocity legend (ordered by prominence)
LEGEND_COUNTRIES = [
    "France","United States","United Kingdom","Germany","Spain","China",
    "Italy","Canada","Switzerland","Japan","Denmark","Brazil","Russia",
    "Australia","Greece","Poland","The Netherlands","Finland","Sweden",
    "India","South Korea","Belgium",
]


## 1. Setup & Data Loading

### Data cleaning and standardisation

The raw CSV files contain known inconsistencies in country naming (e.g. `"China (People's Republic of)"` and `"China"` refer to the same country). Before any aggregation, all country names are mapped to a canonical form via `COUNTRY_NAME_MAP`.

Each institution's data also contains a row for the focal institution itself (self-citations in the OpenCitations graph). Following the same logic as the country-level analysis — which excludes the focal country — we exclude **only the focal institution's own row**, retaining all other Italian institutions. This allows domestic inter-institutional citation relationships to surface in the analysis.

### Data loading and transformation pipeline

The following functions implement the data ingestion process:

- **`normalise_country_names(df)`** — applies the canonical name map to the `country_name` column; called immediately after `read_csv` before any other operation.
- **`load_org_data(institution)`** — reads the two CSV files for one institution (incoming and outgoing), normalises country names, removes the self-citation row, and tags each row with the institution key. Returns `(inbound_df, outbound_df)`.
- **`load_all_available()`** — iterates over all six institutions, calls `load_org_data` for each, and collects results into a single `ALL_DATASETS` dictionary. Institutions whose CSV files are not yet present are skipped with a printed warning, so the notebook runs on partial data without error.


In [2]:
# Country name variants found in the raw data → canonical form
# Mirrors the standardisation applied in the country-level analysis notebook
COUNTRY_NAME_MAP = {
    "China (People's Republic of)": "China",
    "China, People's Republic of":  "China",
    "People's Republic of China":   "China",
    "Hong Kong":                    "Hong Kong SAR",
    "Korea, Republic of":           "South Korea",
    "Korea (Republic of)":          "South Korea",
    "Russian Federation":           "Russia",
}

def normalise_country_names(df: pd.DataFrame) -> pd.DataFrame:
    """Replace known country name variants with their canonical form.
    Applied before any groupby or merge to prevent double-counting."""
    df = df.copy()
    df["country_name"] = df["country_name"].replace(COUNTRY_NAME_MAP)
    return df


def load_org_data(institution: str):
    """
    Load and clean the incoming/outgoing CSVs for one institution.

    Exclusion logic:
      - The focal institution's own row is removed (self-citations).
      - All other Italian institutions are RETAINED, unlike the country-level
        analysis which excludes Italy entirely. This allows domestic
        inter-institutional relationships to appear in the charts.

    Returns: (inbound_df, outbound_df), each tagged with 'institution' key.
    """
    base      = INSTITUTIONS[institution]
    self_name = INSTITUTION_SELF_NAMES[institution]

    inb = normalise_country_names(
        pd.read_csv(base / "citation_counts_organizations_incoming.csv")
    )
    out = normalise_country_names(
        pd.read_csv(base / "citation_counts_organizations_outgoing.csv")
    )

    # Remove only the focal institution's own row
    inb = inb[inb["legal_name"] != self_name].copy()
    out = out[out["legal_name"] != self_name].copy()

    inb["institution"] = institution
    out["institution"] = institution
    return inb, out


def load_all_available() -> dict:
    """
    Load all institutions for which CSV files are present.
    Institutions with missing data are skipped gracefully.
    Returns: {inst_key: (inbound_df, outbound_df)}
    """
    datasets = {}
    for inst in INSTITUTIONS:
        try:
            inb, out = load_org_data(inst)
            datasets[inst] = (inb, out)
            print(f"✓ {inst}: incoming {len(inb):,} orgs · outgoing {len(out):,} orgs")
        except FileNotFoundError:
            print(f"○ {inst}: data not yet available — skipped")
    return datasets


ALL_DATASETS = load_all_available()

# Convenience reference to the primary dataset used in single-institution examples
inbound_df, outbound_df = ALL_DATASETS["UNIBO"]


✓ UNIBO: incoming 72,283 orgs · outgoing 64,128 orgs
✓ UNIMI: incoming 73,539 orgs · outgoing 62,155 orgs
✓ UNIPD: incoming 71,281 orgs · outgoing 63,099 orgs
✓ UNITO: incoming 64,704 orgs · outgoing 57,927 orgs
✓ UPO: incoming 42,285 orgs · outgoing 38,112 orgs
✓ SNS: incoming 27,457 orgs · outgoing 20,714 orgs


## 2. Per-Institution Analysis

### 2a — Combined Butterfly Chart

Each subplot shows the **top-15 partner organisations** ranked by total citation volume (incoming + outgoing combined). Bars extending **left** (gold) = organisations that **cite the focal institution** (incoming). Bars extending **right** (purple) = organisations **cited by the focal institution** (outgoing). Other Italian institutions are included, so domestic relationships surface alongside international ones.


In [3]:
def build_butterfly_data(inb, out, top_n=TOP_N):
    """Merge incoming/outgoing, pick top_n orgs by total, return long-form DataFrame."""
    inb_agg = inb.groupby(["legal_name","country_name","country_code"])["count"].sum().reset_index()
    out_agg = out.groupby(["legal_name","country_name","country_code"])["count"].sum().reset_index()

    merged = pd.merge(
        inb_agg.rename(columns={"count": "incoming"}),
        out_agg.rename(columns={"count": "outgoing"}),
        on=["legal_name","country_name","country_code"], how="outer",
    ).fillna(0)

    merged["total"] = merged["incoming"] + merged["outgoing"]
    top = merged.nlargest(top_n, "total").copy()

    # Y-axis label: truncated name + country
    top["label"] = top.apply(
        lambda r: (r["legal_name"][:36]+"…" if len(r["legal_name"]) > 38 else r["legal_name"])
                  + "  (" + r["country_name"] + ")",
        axis=1,
    )

    inb_long = top[["label","legal_name","country_name","incoming"]].copy()
    inb_long["direction"] = "incoming"
    inb_long["value"]     = -inb_long["incoming"]
    inb_long["count"]     =  inb_long["incoming"]
    inb_long = inb_long.drop(columns="incoming")

    out_long = top[["label","legal_name","country_name","outgoing"]].copy()
    out_long["direction"] = "outgoing"
    out_long["value"]     =  out_long["outgoing"]
    out_long["count"]     =  out_long["outgoing"]
    out_long = out_long.drop(columns="outgoing")

    long_df = pd.concat([inb_long, out_long], ignore_index=True)
    order = top.sort_values("total")["label"].tolist()
    long_df["label"] = pd.Categorical(long_df["label"], categories=order, ordered=True)
    return long_df.sort_values("label")


def plot_butterfly_combined(datasets: dict, top_n: int = TOP_N) -> go.Figure:
    """
    3 × 2 subplot grid — one butterfly chart per institution.
    Each subplot has its own x-axis scale.
    """
    institutions = list(datasets.keys())
    ncols, nrows = 2, 3

    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[INSTITUTION_LABELS[inst] for inst in institutions],
        horizontal_spacing=0.26,
        vertical_spacing=0.055,
    )

    legend_added = set()

    for i, inst in enumerate(institutions):
        row, col = i // ncols + 1, i % ncols + 1
        inb, out = datasets[inst]
        long_df  = build_butterfly_data(inb, out, top_n)
        max_val  = long_df["count"].max()

        for direction, color in [("incoming", COLORS["incoming"]),
                                  ("outgoing",  COLORS["outgoing"])]:
            df_d = long_df[long_df["direction"] == direction]
            show = direction not in legend_added

            fig.add_trace(go.Bar(
                x=df_d["value"],
                y=df_d["label"],
                orientation="h",
                name=direction,
                marker_color=color,
                legendgroup=direction,
                showlegend=show,
                customdata=df_d[["count","direction","country_name","legal_name"]].values,
                hovertemplate=(
                    "<b>%{customdata[3]}</b><br>"
                    "%{customdata[2]}<br>"
                    "Citations: %{customdata[0]:,.0f}<br>"
                    "Direction: %{customdata[1]}<extra></extra>"
                ),
            ), row=row, col=col)

            if show:
                legend_added.add(direction)

        fig.update_xaxes(
            range=[-max_val * 1.12, max_val * 1.12],
            tickformat=",", row=row, col=col,
        )
        fig.add_vline(x=0, line_width=1, line_color="gray", row=row, col=col)

    fig.update_layout(
        template="plotly_white",
        height=nrows * 530,
        barmode="overlay",
        bargap=0.15,
        title_text=f"Top {top_n} Organisations — Incoming vs Outgoing Citations",
        title_x=0.5,
        legend=dict(
            orientation="h", yanchor="bottom", y=1.01,
            xanchor="center", x=0.5,
            title_text="",
        ),
    )
    fig.update_yaxes(tickfont=dict(size=8.5))
    return fig


plot_butterfly_combined(ALL_DATASETS, TOP_N).show()


#### Butterfly Chart — Key Findings

**CNRS is the universal anchor.** It appears at the top of all six subplots with broadly balanced incoming and outgoing bars — the most symmetric high-volume relationship in the dataset.

**Domestic inter-institutional citations are substantial and visible.** Italian universities appear in each other's top-15 across all institutions. The most prominent shared partners are Sapienza University of Rome (in five of six top-15 lists) and University of Bologna and University of Padua (in four each). This confirms that the Italian national research network is a major citation ecosystem in its own right, not a secondary backdrop to international partnerships.

**UNIBO and UNIPD have a strong bilateral tie.** University of Padua ranks 5th for UNIBO and University of Bologna ranks 5th for UNIPD, both with roughly balanced bars — the most symmetric large Italian–Italian pair in the dataset.

**UPO is the most domestically concentrated institution.** Seven of its fifteen top partners are Italian, including Università degli Studi del Piemonte Orientale (its predecessor institution, ranked 1st with the longest bars in the UPO subplot) and University of Turin (ranked 2nd). No other institution shows this degree of domestic top-15 concentration. CERN and IN2P3 are UPO's main international outgoing-skewed partners, with clearly longer purple bars.

**UNIMI's incoming bars are visibly longer than outgoing for IRCCS and CNR.** Italian hospital research networks (IRCCS) and the National Research Council cite UNIMI substantially more than UNIMI cites them back, reflecting UNIMI's role as a primary reference institution for Italian clinical and applied research. Other top partners (Harvard University Press, CSIC) show the typical outgoing bias seen across all institutions.

**SNS has the most symmetric top-15 and the strongest geographic proximity tie.** University of Pisa ranks 3rd for SNS (incoming ≈ outgoing, asymmetry −0.02) — a near-perfectly balanced partnership between two institutions located in the same city. Harvard University Press is absent from SNS's top-15 entirely, replaced by CERN (+0.17) and IN2P3 (+0.03) as the dominant physics infrastructure partners.

**UNITO shows consistent outgoing bias for its largest international partners.** Harvard University Press and Texas Tech University System both show clearly longer outgoing bars, and CERN/IN2P3 appear similarly outgoing-skewed — consistent with UNITO citing physics and computational reference literature more than it is cited back by those institutions.


### 2b — Reciprocity Scatter (Interactive)

Organisations appearing in **both** the top-500 incoming and top-500 outgoing lists for a given institution. Select an institution from the dropdown. The dashed diagonal marks perfect reciprocity — points above cite less back than they are cited; points below cite more than they receive. Bubble size = total citations · colour = country.


In [ ]:
def compute_reciprocity(inb, out, recip_top=RECIP_TOP):
    inb_top = (inb.groupby(["legal_name","country_name"])["count"].sum()
               .reset_index().nlargest(recip_top,"count")
               .rename(columns={"count":"incoming"}))
    out_top = (out.groupby(["legal_name","country_name"])["count"].sum()
               .reset_index().nlargest(recip_top,"count")
               .rename(columns={"count":"outgoing"}))
    recip = pd.merge(inb_top[["legal_name","country_name","incoming"]],
                     out_top[["legal_name","country_name","outgoing"]],
                     on=["legal_name","country_name"], how="inner")
    recip["total"]     = recip["incoming"] + recip["outgoing"]
    recip["asymmetry"] = (recip["outgoing"] - recip["incoming"]) / recip["total"]
    return recip


def plot_reciprocity_interactive(datasets: dict, recip_top: int = RECIP_TOP) -> go.Figure:
    institutions  = list(datasets.keys())
    legend_groups = LEGEND_COUNTRIES + ["Other"]          # "Other" is filterable too
    n_groups      = len(legend_groups)
    block         = n_groups + 1                          # country traces + diagonal
    fig = go.Figure()

    # ── Institution traces (one per country group, + diagonal) ────────────
    for i, inst in enumerate(institutions):
        inb, out = datasets[inst]
        recip    = compute_reciprocity(inb, out, recip_top)
        visible  = (i == 0)

        # size scaling computed on the FULL institution dataset,
        # so bubbles stay comparable across country traces
        t = recip["total"]
        recip = recip.assign(
            size=5 + 32 * (t - t.min()) / (t.max() - t.min() + 1),
            group=recip["country_name"].where(
                recip["country_name"].isin(LEGEND_COUNTRIES), "Other"
            ),
        )

        for country in legend_groups:
            sub = recip[recip["group"] == country]
            fig.add_trace(go.Scatter(
                x=sub["incoming"], y=sub["outgoing"],
                mode="markers",
                name=country,
                legendgroup=country,          # linked to the legend dummy below
                showlegend=False,
                visible=visible,
                marker=dict(
                    size=sub["size"],
                    color=COUNTRY_COLORS.get(country, OTHER_COUNTRY_COLOR),
                    opacity=0.72,
                    line=dict(width=0.5, color="white"),
                ),
                customdata=sub[["legal_name", "country_name",
                                "incoming", "outgoing", "total"]].values,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>%{customdata[1]}<br>"
                    "Incoming: %{customdata[2]:,.0f}<br>"
                    "Outgoing: %{customdata[3]:,.0f}<br>"
                    "Total: %{customdata[4]:,.0f}<extra></extra>"
                ),
            ))

        ax_min = float(min(recip[["incoming", "outgoing"]].min()))
        ax_max = float(max(recip[["incoming", "outgoing"]].max()))
        fig.add_trace(go.Scatter(
            x=[ax_min, ax_max], y=[ax_min, ax_max],
            mode="lines", line=dict(color="gray", dash="dash", width=1),
            visible=visible, showlegend=False, hoverinfo="skip",
        ))

    n_inst_traces = len(institutions) * block

    # ── Country legend traces (always visible, clickable filters) ─────────
    for country in legend_groups:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            name=country,
            marker=dict(color=COUNTRY_COLORS.get(country, OTHER_COUNTRY_COLOR), size=9),
            showlegend=True, visible=True,
            legendgroup=country,              # same group as the data traces
        ))

    # ── Dropdown buttons ──────────────────────────────────────────────────
    buttons = []
    for i, inst in enumerate(institutions):
        vis = [False] * n_inst_traces + [True] * n_groups
        for j in range(block):                # this institution's block
            vis[i * block + j] = True
        buttons.append(dict(
            label=INSTITUTION_LABELS[inst],
            method="update",
            args=[
                {"visible": vis},
                {"title": {"text": (
                    f"Reciprocity Scatter — Top-{recip_top} Bilateral Partners"
                    f"  ({INSTITUTION_LABELS[inst]})<br>"
                    "<sup>Log scale · bubble size = total citations · colour = country"
                    " · click legend to filter countries</sup>"
                ), "x": 0.5}},
            ],
        ))

    fig.update_layout(
        template="plotly_white",
        height=640,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="#FFFFFF", family="Inter, sans-serif', color='#333'"),
        title=dict(text=(
            f"Reciprocity Scatter — Top-{recip_top} Bilateral Partners"
            f"  ({INSTITUTION_LABELS[institutions[0]]})<br>"
            "<sup>Log scale · bubble size = total citations · colour = country</sup>"
        ),
        font=dict(family='Playfair Display, serif', size=18),
        x=0.5),
        xaxis=dict(type="log", title="Incoming citations", gridcolor="rgba(255,255,255,0.07)", zerolinecolor="rgba(255,255,255,0.15)"),
        yaxis=dict(type="log", title="Outgoing citations", gridcolor="rgba(255,255,255,0.07)", zerolinecolor="rgba(255,255,255,0.15)"),
        legend_title_text="Country",
        legend=dict(itemclick="toggleothers", itemdoubleclick="toggle", groupclick="togglegroup"),
        updatemenus=[dict(
            buttons=buttons,
            direction="down",
            showactive=False,
            x=0.0,
            xanchor="left",
            y=1.13,
            yanchor="top",
            bordercolor="rgba(255, 255, 255, 0.2)",
            font=dict(size=12, color="#FFFFFF", family="Inter, sans-serif"),
        )],
        margin=dict(t=120),
    )

    return fig

plot_reciprocity_interactive(ALL_DATASETS, RECIP_TOP).show()
fig = plot_reciprocity_interactive(ALL_DATASETS, RECIP_TOP)

HOVER_CSS = """
<style>
.updatemenu-item-rect:hover { fill: #B7990D !important; }
.updatemenu-item-rect:hover + .updatemenu-item-text,
.updatemenu-button:hover .updatemenu-item-text { fill: #320E3B !important; }
</style>
"""

FONT_CSS = """
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600&family=Playfair+Display:wght@600;700&display=swap" rel="stylesheet">
"""

fragment = fig.to_html(full_html=False, include_plotlyjs="cdn",
                       config={"responsive": True})
html = HOVER_CSS + fragment

try:
    BASE = Path(__file__).resolve().parent.parent
except NameError:
    BASE = Path.cwd()


# OUT_DIR = BASE / ".." / ".." / "website" / "visualizations"
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# OUT_PATH = OUT_DIR / "reciprocity.html"

# with open(OUT_PATH, "w", encoding="utf-8") as f:
#     f.write(FONT_CSS + HOVER_CSS + fig.to_html(full_html=False,
#                                     include_plotlyjs="cdn",
#                                     config={"responsive": True}))

# print(f"Wrote {OUT_PATH}")

Wrote /Users/sara/Documents/GitHub/2025-2026/bloom/map_of_italian_science/data_viz/../../website/visualizations/reciprocity.html


#### Reciprocity Scatter — Key Findings

| Institution | Bilateral partners | Near-diagonal (±0.1) | Above diagonal | Below diagonal |
|---|---|---|---|---|
| UNIBO  | 442 | 64.7% | 31.6% | 3.7% |
| UNIMI  | 442 | 69.0% | 13.8% | 17.2% |
| UNIPD  | 455 | 71.4% | 19.3% | 9.2% |
| UNITO  | 450 | 58.4% | 36.2% | 5.3% |
| UPO    | 455 | 64.0% | 29.9% | 6.2% |
| SNS    | 460 | **73.5%** | 18.3% | 8.3% |

**The majority of bilateral partnerships are balanced.** Across all six institutions, roughly two-thirds of organisations in both top-500 lists sit within ±0.1 of the diagonal. The largest single bubble in every scatter is CNRS — both the most cited and most reciprocal major partner.

**Italian institutions cluster near the diagonal.** With domestic partners included, universities such as Sapienza, University of Padua, and University of Milan appear as mid-sized bubbles in most institution scatters, consistently close to the dashed line. Italian inter-institutional citation exchange is symmetric.

**UNIMI has the largest below-diagonal share (17.2%).** Switching to UNIMI reveals a denser concentration of points below the diagonal than any other institution — Italian hospital and clinical research networks, alongside Chinese biomedical institutions, cite UNIMI's output substantially more than UNIMI cites them back.

**UNITO has the largest above-diagonal share (36.2%).** The UNITO scatter shows a pronounced upward-right drift: many high-volume partners sit clearly above the diagonal. This is the strongest institutional signal of a net citation importer in the dataset.

**SNS achieves the tightest diagonal clustering (73.5% near-diagonal).** Its scatter has the narrowest spread around the dashed line of all six institutions. The above-diagonal region is driven not by publishers but by astrophysics research infrastructure — a pattern unique to SNS.

**UPO's above-diagonal region is driven by physics facilities.** CERN and IN2P3 sit well above the diagonal in UPO's scatter, consistent with the long outgoing bars visible in the butterfly chart. By contrast, UPO's Italian partners (University of Turin, Università degli Studi del Piemonte Orientale) cluster near the diagonal.


---

## 3. Cross-Institution Comparison

### 3a — Proportional Partner Mix

Each bar shows the top-10 partner organisations as a share of that institution's total citation volume (incoming + outgoing combined), stacked to 100%. Comparing bar composition across institutions reveals which partnerships are disproportionately large for a specific institution relative to the others.


In [5]:
def plot_proportional_stacked(datasets: dict, top_n: int = 10) -> go.Figure:
    rows = []
    for inst, (inb, out) in datasets.items():
        inb_agg = inb.groupby(["legal_name","country_name"])["count"].sum().reset_index().rename(columns={"count":"incoming"})
        out_agg = out.groupby(["legal_name","country_name"])["count"].sum().reset_index().rename(columns={"count":"outgoing"})
        m = pd.merge(inb_agg, out_agg, on=["legal_name","country_name"], how="outer").fillna(0)
        m["total"] = m["incoming"] + m["outgoing"]
        m["share"] = m["total"] / m["total"].sum() * 100
        for _, row in m.nlargest(top_n, "total").iterrows():
            rows.append({"institution": INSTITUTION_LABELS[inst],
                         "legal_name": row["legal_name"],
                         "country_name": row["country_name"],
                         "share": row["share"], "total": row["total"]})

    df  = pd.DataFrame(rows)
    fig = px.bar(df, x="institution", y="share", color="legal_name", text="legal_name",
                 hover_data={"total":":,","country_name":True,"share":":.2f"},
                 title=f"Top-{top_n} External Partners as % of Total Citation Volume",
                 labels={"share":"Share of total citations (%)","institution":""})
    fig.update_traces(textposition="inside", textfont_size=7, insidetextanchor="middle")
    fig.update_layout(template="plotly_white", height=540, title_x=0.5,
                      showlegend=False, bargap=0.25,
                      yaxis_title="Share of total citations (%)")
    return fig

plot_proportional_stacked(ALL_DATASETS, top_n=10).show()


#### Proportional Partner Mix — Key Findings

**CNRS dominates the top slot at every institution**, but its share varies meaningfully: it accounts for a larger slice of SNS and UPO's total volume than of UNIBO or UNIMI's, reflecting that smaller and more specialised institutions have more concentrated partner portfolios.

**UPO and SNS show the most concentrated partner structures** — their top-10 partners account for a larger combined share of total citations compared to UNIBO or UNIMI, where citation volume is more spread across a wider range of organisations.

**UNIMI's top-10 is the most distinctive domestically**: Italian hospital networks (IRCCS) and CNR claim slots that at other institutions are occupied by international research universities, reflecting UNIMI's central role in Italy's clinical research ecosystem.


---

## 4. Summary of Findings

### Key institutional patterns

**1. CNRS as the universal anchor.**
CNRS is the top partner by citation volume at all six institutions and maintains near-symmetric citation exchange at each one. It is the only organisation that is simultaneously prominent and balanced across the entire dataset.

**2. Italian inter-institutional networks are substantial.**
With single-institution exclusion in place, domestic partnerships surface clearly. UNIBO–UNIPD form a particularly strong bilateral pair. UNIMI is uniquely embedded in Italy's clinical research network through IRCCS hospital institutions and CNR. UPO is the most domestically concentrated institution: seven of its fifteen top partners are Italian, anchored by a strong bilateral tie with University of Turin.

**3. Each institution has a distinct citation balance.**
UNIMI attracts the most incoming citations relative to outgoing (largest below-diagonal share: 17.2%), driven by Italian and Chinese medical institutions. UNITO generates the most outgoing citations relative to incoming (largest above-diagonal share: 36.2%), acting as the strongest net citation importer. SNS achieves the most symmetric exchange overall (73.5% near-diagonal), and is the only institution where CERN — not Harvard University Press — is the dominant above-diagonal partner.

**4. Harvard University Press is the dominant outgoing sink at five of six institutions** — absent only at SNS. Its asymmetry is highest at UPO (+0.31), the most specialised institution.

**5. Incoming diversity exceeds outgoing diversity at every institution.** Italian universities are cited by a broader range of partner organisations than they actively cite — a structural feature that holds regardless of institution size or disciplinary focus.
